# 02 — Fused Softmax

**Ladder:** Multi-pass naive → per-row fuse → single Triton kernel.

**Study tip.** First count HBM traffic (naive vs fused). Then implement stable softmax with $\max$ subtraction; padded lanes must not spoil $\max$/sum.

Official: [Fused Softmax](https://triton-lang.org/main/getting-started/tutorials/02-fused-softmax.html)

Figures: [`docs/images/`](docs/images/) · visual guide: [`docs/visual-guide.md`](docs/visual-guide.md)

---

# Part I — Mathematics

## 0. Definition, stability, and the softmax Jacobian

Fix a row $x\in\mathbb{R}^{N}$. The **softmax** is

$$
\mathrm{softmax}(x)_i
=
\frac{e^{x_i}}{\displaystyle\sum_{j=0}^{N-1} e^{x_j}},
\qquad i\in[N].
\tag{0.1}
$$

Write $p=\mathrm{softmax}(x)$ and $Z=\sum_j e^{x_j}$ for the partition function.

**Proposition 1 (probability simplex).** $p_i>0$ for all $i$ and
$\sum_{i=0}^{N-1}p_i=1$. Hence $p$ lies in the open simplex
$\Delta^{N-1}=\{p\in\mathbb{R}_{>0}^{N}:\mathbf{1}^{\top}p=1\}$.

*Proof.* Each exponential is positive, so $p_i>0$; summing gives
$\sum_i e^{x_i}/Z=Z/Z=1$. $\square$

**Proposition 2 (shift invariance).** For every $c\in\mathbb{R}$,

$$
\mathrm{softmax}(x-c\mathbf{1})=\mathrm{softmax}(x).
\tag{0.2}
$$

*Proof.* Term by term,

$$
\frac{e^{x_i-c}}{\sum_j e^{x_j-c}}
=
\frac{e^{-c}e^{x_i}}{e^{-c}\sum_j e^{x_j}}
=
\frac{e^{x_i}}{\sum_j e^{x_j}}.
\qquad\square
$$

### 0.1 Numerical stability

Let $m=\max_j x_j$ and choose $c=m$. Then $x_i-m\le0$, hence

$$
0<e^{x_i-m}\le1,
\qquad
\sum_j e^{x_j-m}\in[1,N].
\tag{0.3}
$$

This prevents overflow in $e^{x_i}$ (which occurs for $x_i\gtrsim 88$ in
`float32`) and prevents underflow of the denominator. The stabilized formula is

$$
\mathrm{softmax}(x)_i
=
\frac{\exp(x_i-m)}{\sum_j\exp(x_j-m)}.
\tag{0.4}
$$

If $x_i\ll m$, then $e^{x_i-m}$ may underflow to $0$; this is harmless because
the true probability is below the representable range, and the denominator is
at least $1$.

### 0.2 The Jacobian of softmax

Differentiate (0.1) with respect to $x_j$. With $Z=\sum_k e^{x_k}$,

$$
\frac{\partial p_i}{\partial x_j}
=
\frac{\partial}{\partial x_j}
\left(\frac{e^{x_i}}{Z}\right)
=
\frac{\delta_{ij}e^{x_i}Z-e^{x_i}e^{x_j}}{Z^{2}}
=
p_i\delta_{ij}-p_ip_j .
\tag{0.5}
$$

Hence, in matrix form,

$$
\boxed{\;
J(x)
:=
\frac{\partial p}{\partial x}
=
\operatorname{diag}(p)-pp^{\top}
\;\in\mathbb{R}^{N\times N}.
\;}
\tag{0.6}
$$

**Corollary 3 (row sums vanish).** $J(x)\mathbf{1}=p-p(\mathbf{1}^{\top}p)=0$.

*Proof.* $\operatorname{diag}(p)\mathbf{1}=p$ and $p(p^{\top}\mathbf{1})=p$.
$\square$

This is the manifestation of shift invariance: directions that change all
coordinates equally ($\mathbf{1}$) are in the kernel of $J$. In backward passes
this produces the familiar centering terms in softmax gradients.

**Corollary 4 (symmetry).** $J=J^{\top}$.

*Proof.* $\operatorname{diag}(p)$ is diagonal and $pp^{\top}$ is symmetric.
$\square$

**Spectral fact (optional).** $J$ is symmetric PSD and $J\mathbf{1}=0$. Its
nonzero eigenvalues $\lambda_k$ **interlace** the probabilities,

$$
p_{(k+1)}\le \lambda_k\le p_{(k)},
$$

so the spectrum is controlled by $p$ and vanishes along the constant direction.
This explains why the softmax backward is well conditioned except along that
direction.

### 0.3 Log-sum-exp

The denominator is the **log-sum-exp** (LSE) function

$$
\mathrm{LSE}(x)=\log\sum_j e^{x_j}.
\tag{0.7}
$$

Then $\mathrm{softmax}(x)=\nabla_x\mathrm{LSE}(x)$, because

$$
\frac{\partial}{\partial x_i}\log\sum_j e^{x_j}
=
\frac{e^{x_i}}{\sum_j e^{x_j}}
=
p_i .
\tag{0.8}
$$

The stable evaluation is
$\mathrm{LSE}(x)=m+\log\sum_j e^{x_j-m}$, which is what the fused kernel
computes implicitly through $\ell=\sum_j e^{x_j-m}$.

### 0.4 Worked example

Take $x=(0,1,2)$. Then $m=2$ and

$$
e^{x-m}=(e^{-2},e^{-1},1)\approx(0.1353,\,0.3679,\,1.0000),
\qquad
Z\approx1.5032.
$$

Therefore

$$
p\approx(0.0900,\,0.2447,\,0.6652),
\qquad
\sum_i p_i=1.
$$

The Jacobian is

$$
J=
\begin{pmatrix}
0.0819 & -0.0220 & -0.0599\\
-0.0220 & 0.1848 & -0.1628\\
-0.0599 & -0.1628 & 0.2227
\end{pmatrix},
$$

whose rows sum to zero, as predicted by Corollary 3.

### 0.5 Two-dimensional shape

For $X\in\mathbb{R}^{M\times N}$ softmax acts **row-wise**:

$$
P_{ri}
=
\frac{e^{X_{ri}}}{\sum_{j=0}^{N-1}e^{X_{rj}}},
\qquad r\in[M],\ i\in[N].
\tag{0.9}
$$

Rows are independent, so the Jacobian is block diagonal with blocks
$J(X_{r,:})$. This is why one program can own a row (or a strided set of rows)
with no cross-row communication.

### Visual: theory board (method / logic)

![Whiteboard: multi-pass HBM softmax vs fused one-row kernel](docs/images/benchmark-softmax.png)

*Theory board — five HBM passes (~$8MNb$) vs fused row softmax (~$2MNb$ traffic).*

### Theory board

![Whiteboard: multi-pass HBM softmax vs fused one-row kernel](docs/images/whiteboard-theory.jpg)

*Theory board — five HBM passes (~$8MNb$) vs fused row softmax (~$2MNb$ traffic).*

# Part II — Proof

**Theorem (stable fused softmax).** For each row $x\in\mathbb{R}^{N}$, let
$m=\max_jx_j$, $u_j=e^{x_j-m}$, and $Z=\sum_ju_j$. Then
$u_j/Z=\mathrm{softmax}(x)_j$, and $u_j\in(0,1]$.

*Proof.* Shift invariance gives
$e^{x_j-m}/\sum_ke^{x_k-m}=e^{x_j}/\sum_ke^{x_k}$. Since $x_j-m\le0$, each
exponential lies in $(0,1]$, so the numerator and denominator are finite and
the denominator is at least $1$. $\square$

**Proposition (padding is neutral).** If padded lanes are set to $-\infty$,
then they do not change the row maximum and contribute
$e^{-\infty-m}=0$ to the sum.

*Proof.* $\max(m,-\infty)=m$ and $e^{-\infty}=0$. $\square$

**Corollary (method equivalence).** The multi-pass reference, the per-row
kernel, and the persistent fused kernel compute the same probabilities, up to
floating-point rounding.

*Proof.* Each evaluates the same stabilized formula of the theorem; the
persistent kernel merely reorders rows. $\square$

# Part III — Step-by-step proof

1. **Load the row.** Read $x_j$ for $j<N$; pad $j\ge N$ with $-\infty$.
2. **Row maximum.** Compute $m=\max_jx_j$ over the padded vector; padding is
   neutral.
3. **Center.** Form $x_j-m\le0$; this is the numerically stable shift.
4. **Exponentiate.** Compute $u_j=e^{x_j-m}\in(0,1]$.
5. **Sum.** Compute $Z=\sum_ju_j\ge1$.
6. **Normalize.** Output $u_j/Z$.
7. **Invariants.** $\sum_jp_j=1$ and $p_j\ge0$; padding contributes zero.
8. **Traffic.** The naive chain touches $8MN$ elements; the fused kernel touches
   $2MN$. Hence the large-$N$ speedup ceiling is $4\times$ (equation 1.4).
9. **Jacobian check.** Differentiating the result gives
   $J=\operatorname{diag}(p)-pp^{\top}$, whose rows sum to zero — the algebraic
   signature of step 3.

# Part IV — Coding

The coding part follows the three-rung ladder used in the PDFs: **Method 01 — naive**, **Method 02 — kernel-based**, and **Method 03 — fused kernel-based**. Each method is runnable and is checked against the previous one.


In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import pytest
import torch
import triton
import triton.language as tl
from triton.runtime import driver

def _active_device():
    """Pick Triton active device, else CUDA, else CPU (study helper; no common/ package)."""
    try:
        import triton

        try:
            return triton.runtime.driver.active.get_active_torch_device()
        except Exception:
            pass
    except Exception:
        pass
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


DEVICE = _active_device()


def require_gpu() -> None:
    if not torch.cuda.is_available() and getattr(DEVICE, "type", None) != "cuda":
        pytest.skip(f"GPU not available (active device: {DEVICE})")



def is_hip() -> bool:
    try:
        return triton.runtime.driver.active.get_current_target().backend == "hip"
    except Exception:
        return False


def is_cdna() -> bool:
    try:
        return is_hip() and triton.runtime.driver.active.get_current_target().arch in (
            "gfx940",
            "gfx941",
            "gfx942",
            "gfx90a",
        )
    except Exception:
        return False


### Method 01 — Naive (multi-pass)

Implement softmax as five sequential tensor passes on a matrix
$X\in\mathbb{R}^{M\times N}$ (row-wise):

| Pass | Map | HBM element traffic |
|---|---|---|
| 1 | $m_r=\max_j X_{rj}$ | read $X$: $MN$ |
| 2 | $Z_{rj}=X_{rj}-m_r$ | read $X$, write $Z$: $2MN$ |
| 3 | $U_{rj}=e^{Z_{rj}}$ | read $Z$, write $U$: $2MN$ |
| 4 | $s_r=\sum_j U_{rj}$ | read $U$: $MN$ |
| 5 | $Y_{rj}=U_{rj}/s_r$ | read $U$, write $Y$: $2MN$ |

**Proposition 5 (traffic of the naive chain).** Ignoring the $O(M)$ vectors
$m,s$ and assuming every intermediate is materialized in HBM, the number of
element-sized transfers is

$$
T_{\mathrm{naive}}^{\mathrm{elem}}
=
MN+2MN+2MN+MN+2MN
=
8MN.
\tag{1.1}
$$

With element size $b$, and including the two $M$-vectors written once and read
once each,

$$
T_{\mathrm{naive}}
=
8MNb+4Mb
=
(8N+4)Mb
\ \text{bytes}.
\tag{1.2}
$$

**Fused traffic.** A row-fused kernel reads $X$ once and writes $Y$ once:

$$
T_{\mathrm{fused}}
=
2MNb.
\tag{1.3}
$$

Therefore

$$
\frac{T_{\mathrm{naive}}}{T_{\mathrm{fused}}}
=
\frac{8N+4}{2N}
=
4+\frac{2}{N}
\;\xrightarrow[N\to\infty]{}\;
4.
\tag{1.4}
$$

**Corollary 6 (speedup ceiling).** If both kernels run at the same fraction of
peak bandwidth, fusion can be at most $\approx4\times$ faster in the
large-$N$ limit. It cannot exceed the memory wall; it only removes redundant
traffic.

**FLOPs.** Passes 1 and 4 are reductions ($\approx2MN$ comparisons/adds), pass 2
is $MN$ subtractions, pass 3 is $MN$ exponentials, pass 5 is $MN$ divisions.
The exponential dominates instruction cost but is far cheaper than moving the
tensor: for $b=4$ the intensity is only
$\approx(4MN)/(8MNb)=1/(2b)\approx0.125$ FLOP/byte, so the chain is firmly
memory-bound.

**Concrete check.** For $M=32$, $N=64$, $b=4$:

$$
T_{\mathrm{fused}}=2\cdot32\cdot64\cdot4=16\,\mathrm{KiB},
\qquad
T_{\mathrm{naive}}\approx8\cdot32\cdot64\cdot4=64\,\mathrm{KiB}.
$$

### 1.1 Why not just keep $Z$ and $U$ in cache?

They are $M\times N$ tensors. If $MNb$ exceeds the last-level cache, each pass
re-reads from DRAM. Even when cache-resident, five kernel launches add latency
and prevent overlap of the reduction with the normalization. Fusion removes both
problems at once.


In [ ]:
def naive_softmax(x):
    """
    Numerically stable softmax as a sequence of full-tensor passes.
    (See markdown cells above for the math.)
    """
    # Pass 1: row-wise max (for numerical stability)
    x_max = x.max(dim=1)[0]
    # Pass 2: subtract max
    z = x - x_max[:, None]
    # Pass 3: exp
    numerator = torch.exp(z)
    # Pass 4: row-wise sum
    denominator = numerator.sum(dim=1)
    # Pass 5: normalize
    return numerator / denominator[:, None]


### Method 02 — Kernel-based (row ownership)

For each row index $r$:

$$
\begin{aligned}
m_r &= \max_j X_{rj}, \\
U_{rj} &= \exp(X_{rj}-m_r), \\
s_r &= \sum_j U_{rj}, \\
Y_{rj} &= U_{rj}/s_r.
\end{aligned}
\tag{2.1}
$$

**Claim.** This equals $\mathrm{softmax}$ applied to row $r$ (by §0).

*Proof.* Substitute $c=m_r$ in Proposition 2 and divide. $\square$

Intermediates $m_r,U_r,s_r$ need not be full $M\times N$ global tensors — they
live in registers/SRAM for that row, so the only HBM traffic is one read of the
row and one write of the result.

**Teaching shape.** $X\in\mathbb{R}^{32\times64}$: each "program" conceptually
walks one row end-to-end before moving on.

### 2.1 Splitting a row: the online-softmax recurrence

A row may also be processed in key blocks
$B_1,B_2,\dots,B_T\subseteq[N]$ (disjoint, covering $[N]$), which is exactly what
attention needs. Let $s^{(t)}=X_{:,\,B_t}\in\mathbb{R}^{B_N}$ be the scores of
block $t$ and $v^{(t)}\in\mathbb{R}^{B_N\times D}$ the corresponding values.
Maintain

$$
m^{(t)}\in\mathbb{R},
\qquad
\ell^{(t)}\in\mathbb{R},
\qquad
\mathrm{Acc}^{(t)}\in\mathbb{R}^{D}.
$$

**Initialization.** $m^{(0)}=-\infty$, $\ell^{(0)}=0$,
$\mathrm{Acc}^{(0)}=0$.

**Update.**

$$
\begin{aligned}
m^{(t)}
&=
\max\!\left(m^{(t-1)},\ \max_j s^{(t)}_j\right),\\[2pt]
\ell^{(t)}
&=
e^{\,m^{(t-1)}-m^{(t)}}\,\ell^{(t-1)}
+
\sum_j e^{\,s^{(t)}_j-m^{(t)}},\\[2pt]
\mathrm{Acc}^{(t)}
&=
e^{\,m^{(t-1)}-m^{(t)}}\,\mathrm{Acc}^{(t-1)}
+
\sum_j e^{\,s^{(t)}_j-m^{(t)}}\,v^{(t)}_j .
\end{aligned}
\tag{2.2}
$$

**Finalize.**

$$
O=\frac{\mathrm{Acc}^{(T)}}{\ell^{(T)}}.
\tag{2.3}
$$

**Theorem 7 (online softmax is exact).** For every $t\ge0$,

$$
m^{(t)}=\max_{u\le t}\ \max_{j\in B_u}s_j,
\qquad
\ell^{(t)}=\sum_{u\le t}\sum_{j\in B_u}e^{\,s_j-m^{(t)}},
\qquad
\mathrm{Acc}^{(t)}=\sum_{u\le t}\sum_{j\in B_u}e^{\,s_j-m^{(t)}}v_j .
\tag{2.4}
$$

*Proof by induction.* The base case $t=0$ is the empty sum. Assume (2.4) for
$t-1$ and let $\mu=m^{(t)}$. By definition $\mu=\max(m^{(t-1)},\max_j s^{(t)}_j)$,
which proves the first identity. For the second,

$$
\begin{aligned}
\ell^{(t)}
&=
e^{m^{(t-1)}-\mu}
\underbrace{\sum_{u\le t-1}\sum_{j\in B_u}e^{s_j-m^{(t-1)}}}_{=\,\ell^{(t-1)}}
+
\sum_{j\in B_t}e^{s_j-\mu}\\
&=
\sum_{u\le t-1}\sum_{j\in B_u}e^{s_j-\mu}
+
\sum_{j\in B_t}e^{s_j-\mu}
=
\sum_{u\le t}\sum_{j\in B_u}e^{s_j-\mu}.
\end{aligned}
$$

The same computation with the factor $v_j$ appended proves the third identity.
$\square$

**Corollary 8 (final normalization).**

$$
O
=
\frac{\sum_{j\in[N]}e^{s_j-m^{(T)}}v_j}
{\sum_{j\in[N]}e^{s_j-m^{(T)}}}
=
\sum_{j\in[N]}\mathrm{softmax}(s)_j\,v_j .
\tag{2.5}
$$

*Proof.* Divide numerator and denominator by $e^{-m^{(T)}}$; the common factor
cancels, leaving the definition of softmax. $\square$

**Numerical detail.** On the first block $m^{(0)}=-\infty$, so
$e^{m^{(0)}-m^{(1)}}=e^{-\infty}=0$; the old $\ell$ and $\mathrm{Acc}$ are
correctly discarded. The implementation computes `alpha = exp(m_i - m_new)`,
which is $0$ in that case and $\le 1$ afterwards, because
$m_{\text{new}}\ge m_{\text{old}}$. Thus `alpha` can only shrink the
accumulator — it never causes
overflow.

**Cost.** The recurrence stores $O(D)$ state per row, so the total state is
$O(ND)=O(N)$ for fixed $D$ — instead of the naive $O(N^2)$ score matrix. That is
the memory property FlashAttention exploits.


In [ ]:
def row_softmax_pseudo(x):
    """
    Same math as naive_softmax, but structured like the Triton kernel:
    (See markdown cells above for the math.)
    """
    M, N = x.shape
    y = torch.empty_like(x)

    for row_idx in range(M):
        row = x[row_idx]  # one logical load of the row
        row_max = row.max()
        numer = torch.exp(row - row_max)
        denom = numer.sum()
        y[row_idx] = numer / denom  # one logical store of the row

    return y


#### Direct Triton kernel

The Python per-row model above is the algorithm; the kernel below gives **one
program per row** and evaluates the stabilized formula entirely in registers.
It is the direct kernel translation of the naive multi-pass method, without the
persistent/occupancy-aware scheduling used by Method 03.


In [ ]:
@triton.jit
def softmax_row_kernel(
    output_ptr,
    input_ptr,
    input_row_stride,
    output_row_stride,
    n_cols,
    BLOCK_SIZE: tl.constexpr,
):
    """Method 02: one program per row, straight-line stable softmax."""
    row = tl.program_id(0)
    cols = tl.arange(0, BLOCK_SIZE)
    mask = cols < n_cols
    # -inf makes padded lanes neutral for both max and exp/sum.
    x = tl.load(
        input_ptr + row * input_row_stride + cols,
        mask=mask,
        other=-float("inf"),
    )
    m = tl.max(x, axis=0)
    numerator = tl.exp(x - m)
    denominator = tl.sum(numerator, axis=0)
    y = numerator / denominator
    tl.store(output_ptr + row * output_row_stride + cols, y, mask=mask)


def softmax_kernel_based(x):
    n_rows, n_cols = x.shape
    y = torch.empty_like(x)
    block = triton.next_power_of_2(n_cols)
    # Grid size n_rows: one program for each row.
    softmax_row_kernel[(n_rows,)](
        y, x, x.stride(0), y.stride(0), n_cols, BLOCK_SIZE=block
    )
    return y


### Method 03 — Fused kernel-based (persistent)

Program $\mathrm{pid}$ walks rows $\mathrm{pid},\mathrm{pid}+P,\mathrm{pid}+2P,\dots$
where $P=\texttt{tl.num\_programs}(0)$ (a persistent row schedule). For each row,
the stable formula of §0 is evaluated entirely in-register after one masked
`tl.load`.

**Why a persistent grid?** Launching $M$ programs is wasteful when $M$ is huge
and each row is short: the grid is capped at the number of resident programs
that the occupancy calculation allows. The loop

```python
for row_idx in tl.range(row_start, n_rows, row_step, num_stages=num_stages):
```

is exactly the stride schedule from notebook 09:

$$
\mathrm{row}_k=\mathrm{pid}+kP,
\qquad k=0,1,\dots
\tag{3.1}
$$

and by the coverage lemma it visits every row exactly once.

### 3.1 Block size and padding

$$
B=2^{\lceil\log_2 N\rceil},
\qquad
\tilde{X}_{rj}=
\begin{cases}
X_{rj}, & j<N,\\
-\infty, & N\le j<B.
\end{cases}
\tag{3.2}
$$

Then

$$
\max_{j<B}\tilde{X}_{rj}=\max_{j<N}X_{rj},
\qquad
e^{-\infty-m}=0,
\tag{3.3}
$$

so padding contributes neither to the max nor to the sum. This is neutral-element
selection (2.6) in action. Using `other=0.0` here would add $B-N$ terms
$e^{-m}>0$ and bias every probability downward.

### 3.2 Complexity and bandwidth

The kernel performs

$$
\text{reads}=MN,
\qquad
\text{writes}=MN,
\qquad
\text{FLOPs}\approx 4MN,
\tag{3.4}
$$

so

$$
\mathrm{GB/s}\approx\frac{2MNb}{t\cdot10^{9}}
\tag{3.5}
$$

as in (1.3). The arithmetic intensity is $\approx 4/(2b)=2/b$ FLOP/byte, still
far below machine balance, so the kernel is bandwidth-bound and the speedup over
the naive chain tends to $4\times$ (1.4).

### 3.3 Occupancy-aware grid

The launcher warms the kernel up, reads `n_regs` and shared-memory usage, and
estimates how many programs fit per SM:

$$
\text{occupancy}
=
\min\!\left(
\frac{\text{regs per SM}}{\text{regs per program}\cdot W\cdot\text{warps}},
\ \frac{\text{SMEM per SM}}{\text{SMEM per program}}
\right),
\tag{3.6}
$$

$$
P=\min(\text{SMs}\cdot\text{occupancy},\,M).
\tag{3.7}
$$

Too few programs underutilize the GPU; too many cause tail effects and cache
thrashing. Persistence lets us choose $P$ independently of $M$.

| Stage | Program owns | Memory pattern |
|---|---|---|
| `naive_softmax` | whole tensor (many passes) | $\sim8MN$ element trips |
| `row_softmax_pseudo` | one row at a time | intermediates local |
| `softmax_kernel` | strided set of rows | one load + one store per row |

### 3.4 Numerical error

For a row, the computed probabilities satisfy

$$
\hat p_i
=
p_i\big(1+O(u)\big),
\qquad
u=2^{-24}\ \text{(fp32)},
\tag{3.8}
$$

because the max subtraction is exact, the exponential is correctly rounded to
within a few ULP, and the sum of $N$ nonnegative terms has relative error
$O(Nu)$ in the worst case and $O(\sqrt{N}u)$ in the statistical case. Casting
the result to `float16` adds a final relative error $O(2^{-11})$, which is why
GPU comparisons use loose tolerances.


In [ ]:
@triton.jit
def softmax_kernel(
    output_ptr,
    input_ptr,
    input_row_stride,
    output_row_stride,
    n_rows,
    n_cols,
    BLOCK_SIZE: tl.constexpr,
    num_stages: tl.constexpr,
):
    # Persistent programs: each pid walks a strided set of rows
    # (same "one row at a time" idea as row_softmax_pseudo).
    row_start = tl.program_id(0)
    row_step = tl.num_programs(0)
    for row_idx in tl.range(row_start, n_rows, row_step, num_stages=num_stages):
        row_start_ptr = input_ptr + row_idx * input_row_stride
        col_offsets = tl.arange(0, BLOCK_SIZE)
        input_ptrs = row_start_ptr + col_offsets
        mask = col_offsets < n_cols
        # Pad OOB with -inf so max/exp ignore them
        row = tl.load(input_ptrs, mask=mask, other=-float("inf"))
        row_minus_max = row - tl.max(row, axis=0)
        numerator = tl.exp(row_minus_max)
        denominator = tl.sum(numerator, axis=0)
        softmax_output = numerator / denominator
        output_row_start_ptr = output_ptr + row_idx * output_row_stride
        output_ptrs = output_row_start_ptr + col_offsets
        tl.store(output_ptrs, softmax_output, mask=mask)


def _device_props():
    """Lazy so importing this module on CPU (no Triton driver) still works for pseudo tests."""
    properties = driver.active.utils.get_device_properties(DEVICE.index)
    return properties


def softmax(x):
    properties = _device_props()
    NUM_SM = properties["multiprocessor_count"]
    NUM_REGS = properties["max_num_regs"]
    SIZE_SMEM = properties["max_shared_mem"]
    WARP_SIZE = properties["warpSize"]

    n_rows, n_cols = x.shape
    BLOCK_SIZE = triton.next_power_of_2(n_cols)
    num_warps = 8
    num_stages = 4 if SIZE_SMEM > 200000 else 2
    y = torch.empty_like(x)
    # Warmup to read register / smem usage, then pick occupancy-aware grid
    kernel = softmax_kernel.warmup(
        y,
        x,
        x.stride(0),
        y.stride(0),
        n_rows,
        n_cols,
        BLOCK_SIZE=BLOCK_SIZE,
        num_stages=num_stages,
        num_warps=num_warps,
        grid=(1,),
    )
    kernel._init_handles()
    n_regs = kernel.n_regs
    size_smem = kernel.metadata.shared
    if is_hip():
        num_gprs = NUM_REGS * 2 if is_cdna() else NUM_REGS
        max_num_threads = properties["max_threads_per_sm"]
        max_num_waves = max_num_threads // WARP_SIZE
        occupancy = min(num_gprs // WARP_SIZE // n_regs, max_num_waves) // num_warps
    else:
        occupancy = NUM_REGS // (n_regs * WARP_SIZE * num_warps)
    occupancy = min(occupancy, SIZE_SMEM // size_smem)
    num_programs = min(NUM_SM * occupancy, n_rows)
    kernel[(num_programs, 1, 1)](
        y, x, x.stride(0), y.stride(0), n_rows, n_cols, BLOCK_SIZE, num_stages
    )
    return y


### Pitfalls

- Softmax without subtracting the max → overflow in `exp`.
- Multi-pass PyTorch looks simple but multiplies HBM traffic.
- `BLOCK_SIZE` must cover $N$ (`next_power_of_2`); mask invalid lanes as $-\infty$ for the max.
- Using `other=0` for padded lanes: then $e^{0-m}$ contributes spurious mass.

### Self-check

1. Derive the Jacobian (0.6) and verify its rows sum to zero.
2. Prove $\mathrm{softmax}(x)=\nabla\mathrm{LSE}(x)$.
3. Derive traffic $\approx8MNb$ vs $\approx2MNb$ for naive vs fused.
4. When would fusion not help? (tiny rows already cache-resident)


In [ ]:
def test_pseudo_matches_torch() -> None:
    """CPU-runnable: naive and per-row pseudo match torch.softmax."""
    torch.manual_seed(0)
    x = torch.randn(32, 64)
    ref = torch.softmax(x, dim=1)
    assert torch.allclose(naive_softmax(x), ref, atol=1e-5, rtol=1e-5)
    assert torch.allclose(row_softmax_pseudo(x), ref, atol=1e-5, rtol=1e-5)


def test_correctness() -> None:
    require_gpu()
    torch.manual_seed(0)
    x = torch.randn(1823, 781, device=DEVICE)
    y_triton = softmax(x)
    y_torch = torch.softmax(x, axis=1)
    assert torch.allclose(y_triton, y_torch)
    print("correctness OK")


In [ ]:
@triton.testing.perf_report(
    triton.testing.Benchmark(
        x_names=["N"],
        x_vals=[128 * i for i in range(2, 40)],
        line_arg="provider",
        line_vals=["triton", "torch", "naive_softmax"],
        line_names=["Triton", "Torch", "Naive Softmax"],
        styles=[("blue", "-"), ("green", "-"), ("red", "-")],
        ylabel="GB/s",
        plot_name="softmax-performance",
        args={"M": 4096},
    )
)
def benchmark(M, N, provider):
    x = torch.randn(M, N, device=DEVICE, dtype=torch.float32)
    if provider == "torch":
        ms = triton.testing.do_bench(lambda: torch.softmax(x, axis=-1))
    if provider == "triton":
        ms = triton.testing.do_bench(lambda: softmax(x))
    if provider == "naive_softmax":
        ms = triton.testing.do_bench(lambda: naive_softmax(x))
    # Naive softmax moves 8MNb (5 reads + 3 writes); the fused paths move 2MNb.
    traffic = 8 if provider == "naive_softmax" else 2
    gbps = lambda ms: traffic * x.numel() * x.element_size() * 1e-9 / (ms * 1e-3)
    return gbps(ms)


In [ ]:
def test_kernel_methods() -> None:
    """Method 01 (multi-pass) vs Method 02 (row kernel) vs Method 03 (fused)."""
    require_gpu()
    torch.manual_seed(0)
    x = torch.randn(257, 1000, device=DEVICE)
    ref = torch.softmax(x, dim=1)
    assert torch.allclose(naive_softmax(x), ref, atol=1e-5, rtol=1e-5)
    assert torch.allclose(
        softmax_kernel_based(x), ref, atol=1e-5, rtol=1e-5
    )
    assert torch.allclose(softmax(x), ref, atol=1e-5, rtol=1e-5)
    print("softmax: methods 01, 02 and 03 OK")
